In [1]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")

In [2]:
def geocode(numero=None, rue=None, ville=None, code_postal=None):
    """Adresse texte -> adresse normalisée + coordonnées."""
    index = f"bano-{code_postal[:2]}" if code_postal else "bano"
    should = []
    if numero:
        should.append({"match": {"address.number": numero}})
    if rue:
        should.append({"match": {"address.street_name": rue}})
    if ville:
        should.append({"match": {"address.city": ville}})

    r = es.search(index=index, size=1, query={"bool": {"should": should}})
    hits = r["hits"]["hits"]
    if not hits:
        return None
    return {**hits[0]["_source"], "_score": hits[0]["_score"]}

In [3]:
def reverse_geocode(lat, lon, rayon="1km"):
    """Point GPS -> adresse la plus proche."""
    r = es.search(
        index="bano",
        size=1,
        query={"bool": {"filter": {
            "geo_distance": {"distance": rayon, "location": {"lat": lat, "lon": lon}}
        }}},
        sort=[{"_geo_distance": {"location": {"lat": lat, "lon": lon}}}],
    )
    hits = r["hits"]["hits"]
    if not hits:
        return None
    return {**hits[0]["_source"], "distance_m": round(hits[0]["sort"][0], 1)}

In [5]:
import elasticsearch
print(elasticsearch.__version__)   # doit afficher (8, x, y)

(8, 19, 3)


In [6]:
print(geocode(numero="9", rue="r coulets", ville="abbans"))


{'address': {'city': 'Abbans-Dessous', 'number': '9', 'zipcode': '25320', 'street_name': 'Rue des Coulets'}, 'region': '25', 'source': 'OSM', 'id': '250010015-9', 'location': {'lon': 5.871687, 'lat': 47.1332}, '_score': 19.643538}


In [7]:
print(reverse_geocode(47.2378, 6.0241))

{'address': {'city': 'Besançon', 'number': '52', 'zipcode': '25000', 'street_name': 'Grande Rue'}, 'region': '25', 'source': 'OSM', 'id': '250562440-52', 'location': {'lon': 6.02403, 'lat': 47.237805}, 'distance_m': 5.3}


In [8]:
def geocode_batch(adresses):
    body = []
    for a in adresses:
        body.append({"index": "bano"})
        body.append({"size": 1, "query": {"bool": {"should": [
            {"match": {"address.street_name": a["rue"]}},
            {"match": {"address.city": a["ville"]}},
        ]}}})
    res = es.msearch(searches=body)
    return [r["hits"]["hits"][0]["_source"] if r["hits"]["hits"] else None
            for r in res["responses"]]